# BASS vs. Fixed-Basis vs. MPS

Empirical comparison across the five circuit families defined in the
manuscript (Sec. VI.A): 1D brickwork, 2D brickwork, RFIM (W=2), QAOA.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np


def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for c in (start, *start.parents):
        if (c / "src").is_dir() and (c / "requirements.txt").exists():
            return c
    raise RuntimeError("Repo root not found")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

DATA_DIR = REPO_ROOT / "tests" / "results"
FIG_DIR = REPO_ROOT / "tests" / "figures"
DATA_DIR.mkdir(exist_ok=True, parents=True)
FIG_DIR.mkdir(exist_ok=True, parents=True)

from src.simulation.bass_simulator import BASS
from src.simulation.simulator import FixedBasisSimulator
from src.simulation.mps_simulator import SimpleMPS
from src.core.seeding import ExperimentSeeder
from src.circuits.families import FAMILIES
from src.experimentation.runner import exact_statevector, fidelity_bass
from src.analysis.stats import (
    trial_stats,
    paired_comparison_stats,
    build_significance_table,
    format_significance_table,
)
from src.analysis.results_io import TrialRecord, TrialLedger
from src.benchmarking.fidelity import compute_fidelity
from src.visualization.style import mplstyle
import matplotlib.pyplot as plt

mplstyle()
print("Imports OK. Repo root:", REPO_ROOT)

Imports OK. Repo root: c:\Users\karti\pyth\BASS


In [ ]:
# Configuration
QUICK_TEST = False  # flip to True first to sanity-check the pipeline quickly

BASS_KW = dict(optimize_every=5, buffer_factor=1, use_2qubit_rotations=False)
EXPERIMENT_ENTROPY = 20260610  # log this -- it reproduces every seed below
N_BOOT = 4000

MPS_FAMILY_NAMES = ["1D Brickwork", "2D Brickwork", "RFIM (W=2)", "QAOA (p=3)"]

if QUICK_TEST:
    N_TRIALS = 15
    K_VALUES = [200, 2000]
    N_OVERRIDE = {
        "1D Brickwork": 10,
        "2D Brickwork": 6,
        "RFIM (W=2)": 10,
        "QAOA (p=3)": 10,
        "UCCSD": 8,
    }
else:
    N_TRIALS = 100
    K_VALUES = [500, 2000, 5000, 10000, 20000]
    N_OVERRIDE = {}  # use each family's registered default_N

exp = ExperimentSeeder(entropy=EXPERIMENT_ENTROPY)

CIRCUIT_FAMILIES = {
    name: {
        "N": N_OVERRIDE.get(name, FAMILIES[name].default_N),
        "gen": FAMILIES[name].generator,
    }
    for name in MPS_FAMILY_NAMES
}

NPZ_PATH = DATA_DIR / (
    "mps_comparison_quick.npz" if QUICK_TEST else "mps_comparison.npz"
)
META_PATH = NPZ_PATH.with_suffix(".meta.json")
EXPECTED_META = {
    "N_TRIALS": N_TRIALS,
    "K_VALUES": K_VALUES,
    "entropy": exp.entropy,
    "families": {name: spec["N"] for name, spec in CIRCUIT_FAMILIES.items()},
}
print(f"Experiment entropy (log this): {exp.entropy}")
print(f"N_TRIALS={N_TRIALS}, k in {K_VALUES}")
print("Families:", {n: s["N"] for n, s in CIRCUIT_FAMILIES.items()})

Experiment entropy (log this): 20260610
N_TRIALS=100, k in [500, 2000, 5000, 10000, 20000]
Families: {'1D Brickwork': 20, '2D Brickwork': 20, 'RFIM (W=2)': 20, 'QAOA (p=3)': 16}


In [ ]:
# Run (or load cached) sweep. Each simulator is instantiated and simulate()'d
# exactly once per trial. Every trial is also recorded
# into a TrialLedger for full per-trial, per-seed traceability.
def run_or_load():
    ledger = TrialLedger()

    if NPZ_PATH.exists() and META_PATH.exists():
        stored_meta = json.loads(META_PATH.read_text())
        if stored_meta == EXPECTED_META:
            print("Loaded from cache:", NPZ_PATH)
            trials_csv = DATA_DIR / NPZ_PATH.name.replace(".npz", "_trials.csv")
            if trials_csv.exists():
                ledger = TrialLedger.from_csv(trials_csv)
            return dict(np.load(NPZ_PATH, allow_pickle=True)), ledger
        print("Cache meta mismatch -- re-running.")

    out = {}
    fam_names = list(CIRCUIT_FAMILIES.keys())
    for fi, fam_name in enumerate(fam_names):
        spec = CIRCUIT_FAMILIES[fam_name]
        Nq, gen = spec["N"], spec["gen"]
        for ki, k in enumerate(K_VALUES):
            chi = SimpleMPS.chi_for_k(Nq, k)
            f_fixed = np.full(N_TRIALS, np.nan)
            f_bass = np.full(N_TRIALS, np.nan)
            f_mps = np.full(N_TRIALS, np.nan)
            for t in range(N_TRIALS):
                seed = exp.seq(fam_name, k, t)
                rng = np.random.default_rng(seed)
                circuit = gen(Nq, rng)
                exact_sv = exact_statevector(Nq, circuit)

                fixed_sim = FixedBasisSimulator(Nq, k, verbose=False)
                fixed_state = fixed_sim.simulate(circuit)
                f_fixed[t] = compute_fidelity(fixed_state, exact_sv)

                bass_sim = BASS(Nq, k, **BASS_KW)
                bass_state = bass_sim.simulate(circuit)
                f_bass[t] = fidelity_bass(bass_sim, bass_state, exact_sv)

                mps = SimpleMPS(Nq, chi)
                mps.simulate(circuit)
                f_mps[t] = mps.fidelity(exact_sv)

                for method, fid in [
                    ("Fixed-basis", f_fixed[t]),
                    ("BASS", f_bass[t]),
                    ("MPS", f_mps[t]),
                ]:
                    ledger.add(
                        TrialRecord(
                            family=fam_name,
                            method=method,
                            N=Nq,
                            k=k,
                            chi=chi if method == "MPS" else None,
                            trial=t,
                            seed_entropy=exp.entropy,
                            fidelity=fid,
                        )
                    )

            key = f"{fam_name}||{k}"
            out[f"{key}||f_fixed"] = f_fixed
            out[f"{key}||f_bass"] = f_bass
            out[f"{key}||f_mps"] = f_mps
            print(
                f"  {fam_name:14s} k={k:6d} (chi={chi:3d})  "
                f"med(F_fixed)={np.median(f_fixed):.3e}  "
                f"med(F_bass)={np.median(f_bass):.3e}  "
                f"med(F_mps)={np.median(f_mps):.3e}"
            )

    np.savez(NPZ_PATH, **out)
    META_PATH.write_text(json.dumps(EXPECTED_META, indent=2))
    trials_csv = DATA_DIR / NPZ_PATH.name.replace(".npz", "_trials.csv")
    trials_npz = DATA_DIR / NPZ_PATH.name.replace(".npz", "_trials.npz")
    ledger.to_csv(trials_csv)
    ledger.to_npz(trials_npz)
    print("Saved:", NPZ_PATH, trials_csv, trials_npz)
    return out, ledger


data, ledger = run_or_load()
print(f"Per-trial ledger: {len(ledger)} rows")

  1D Brickwork   k=   500 (chi=  5)  med(F_fixed)=7.145e-07  med(F_bass)=3.333e-04  med(F_mps)=2.277e-01
  1D Brickwork   k=  2000 (chi= 10)  med(F_fixed)=3.733e-06  med(F_bass)=7.825e-04  med(F_mps)=5.804e-01
  1D Brickwork   k=  5000 (chi= 16)  med(F_fixed)=4.046e-05  med(F_bass)=2.621e-03  med(F_mps)=8.451e-01
  1D Brickwork   k= 10000 (chi= 22)  med(F_fixed)=2.252e-04  med(F_bass)=6.362e-03  med(F_mps)=8.895e-01
  1D Brickwork   k= 20000 (chi= 32)  med(F_fixed)=1.180e-03  med(F_bass)=1.808e-02  med(F_mps)=9.589e-01
  2D Brickwork   k=   500 (chi=  5)  med(F_fixed)=2.749e-04  med(F_bass)=2.505e-02  med(F_mps)=3.554e-02
  2D Brickwork   k=  2000 (chi= 10)  med(F_fixed)=2.236e-03  med(F_bass)=4.458e-02  med(F_mps)=8.653e-02
  2D Brickwork   k=  5000 (chi= 16)  med(F_fixed)=8.866e-03  med(F_bass)=7.761e-02  med(F_mps)=1.621e-01
  2D Brickwork   k= 10000 (chi= 22)  med(F_fixed)=2.668e-02  med(F_bass)=1.161e-01  med(F_mps)=2.114e-01
  2D Brickwork   k= 20000 (chi= 32)  med(F_fixed)=6.412

In [ ]:
# significance tables: BASS vs. Fixed-basis, BASS vs. MPS
records_vs_fixed, records_vs_mps = [], []
for fam_name in CIRCUIT_FAMILIES:
    for k in K_VALUES:
        key = f"{fam_name}||{k}"
        f_fixed = data[f"{key}||f_fixed"]
        f_bass = data[f"{key}||f_bass"]
        f_mps = data[f"{key}||f_mps"]
        label = f"{fam_name}, k={k}"
        records_vs_fixed.append(
            paired_comparison_stats(f_bass, f_fixed, label=label, n_boot=N_BOOT)
        )
        records_vs_mps.append(
            paired_comparison_stats(f_bass, f_mps, label=label, n_boot=N_BOOT)
        )

df_vs_fixed, meta_vs_fixed = build_significance_table(records_vs_fixed)
df_vs_mps, meta_vs_mps = build_significance_table(records_vs_mps)

print("=== BASS vs. Fixed-Basis ===")
print(format_significance_table(df_vs_fixed, meta_vs_fixed))
print()
print("=== BASS vs. MPS (chi = round(sqrt(k/N))) ===")
print(format_significance_table(df_vs_mps, meta_vs_mps))

df_vs_fixed.to_csv(DATA_DIR / "mps_comparison_vs_fixed.csv", index=False)
df_vs_mps.to_csv(DATA_DIR / "mps_comparison_vs_mps.csv", index=False)
print("\nSaved CSVs to", DATA_DIR)

=== BASS vs. Fixed-Basis ===
Paired comparisons: 20 tests (holm-bonferroni on permutation_log_p_raw, family-wise alpha=0.05); 0 skipped (insufficient data)
----------------------------------------------------------------------------------------------------
                label   n              median_method [IQR]            median_baseline [IQR] GM ratio [95% CI] win rate [Wilson 95% CI]  Wilcoxon p (log, raw)  Permutation p (log, raw)  Wilcoxon p (raw-scale, legacy)  p (permutation_log_p_raw, Holm)  reject H0  effect size r  n_excluded_zero
  1D Brickwork, k=500 100 3.333e-04 [1.307e-04, 8.942e-04] 7.145e-07 [2.317e-07, 2.548e-06]    466 [299, 738]     100/100 [0.96, 1.00]           1.948280e-18                   0.00005                    1.948280e-18                            0.001       True          1.000                0
 1D Brickwork, k=2000 100 7.825e-04 [4.261e-04, 1.369e-03] 3.733e-06 [1.357e-06, 1.267e-05]    204 [135, 309]     100/100 [0.96, 1.00]           1.948280e-18  